# Model Parameter Count Calculations

This notebook computes total and trainable parameter counts for the five model families used in Phases 2 and 3.

The current implementation targets a comparable ~25M parameter regime across families.
All counts are measured directly from instantiated adapters.

In [4]:
import pandas as pd

from speech_recognition.config import ModelConfig
from speech_recognition.models.registry import build_model_adapter

In [5]:
families = ["ast", "convnext", "ssamba", "xlstm", "mlp_mixer"]

# ~25M-target configs for AST/SSAMBA/xLSTM; ConvNeXt and MLP-Mixer unchanged.
family_configs = {
    "ast": ModelConfig(
        family="ast",
        num_classes=12,
        pretrained=False,
        ast_hidden_size=512,
        ast_num_hidden_layers=8,
        ast_num_attention_heads=8,
        ast_intermediate_size=2048,
    ),
    "convnext": ModelConfig(family="convnext", num_classes=12, pretrained=False),
    "ssamba": ModelConfig(
        family="ssamba",
        num_classes=12,
        pretrained=False,
        ssamba_d_model=768,
        ssamba_d_state=64,
        ssamba_expand=2,
        ssamba_num_layers=6,
    ),
    "xlstm": ModelConfig(
        family="xlstm",
        num_classes=12,
        pretrained=False,
        xlstm_dim=704,
        xlstm_num_blocks=8,
    ),
    "mlp_mixer": ModelConfig(family="mlp_mixer", num_classes=12, pretrained=False),
}

rows = []
for family in families:
    config = family_configs[family]
    adapter = build_model_adapter(family=family, model_config=config)
    total_params = sum(p.numel() for p in adapter.parameters())
    trainable_params = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
    rows.append(
        {
            "model": family,
            "source_library": adapter.source_library,
            "backbone_class": adapter.backbone.__class__.__name__,
            "count_method": "measured",
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "ast_hidden_size": config.ast_hidden_size if family == "ast" else None,
            "ast_num_hidden_layers": config.ast_num_hidden_layers if family == "ast" else None,
            "ast_num_attention_heads": config.ast_num_attention_heads if family == "ast" else None,
            "ast_intermediate_size": config.ast_intermediate_size if family == "ast" else None,
            "ssamba_d_model": config.ssamba_d_model if family == "ssamba" else None,
            "ssamba_d_state": config.ssamba_d_state if family == "ssamba" else None,
            "ssamba_num_layers": config.ssamba_num_layers if family == "ssamba" else None,
            "xlstm_dim": config.xlstm_dim if family == "xlstm" else None,
            "xlstm_num_blocks": config.xlstm_num_blocks if family == "xlstm" else None,
        }
    )

df = pd.DataFrame(rows)
df

Mamba SSM macOS: Running on Apple Silicon with MPS acceleration


,model,source_library,backbone_class,count_method,total_parameters,trainable_parameters,ast_hidden_size,ast_num_hidden_layers,ast_num_attention_heads,ast_intermediate_size,ssamba_d_model,ssamba_d_state,ssamba_num_layers,xlstm_dim,xlstm_num_blocks
0,ast,transformers,ASTBackboneWrapper,measured,25416204,25416204,512.0,8.0,8.0,2048.0,NaN,NaN,NaN,NaN,NaN
1,convnext,torchvision,ConvNeXt,measured,27826284,27826284,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ssamba,mamba-ssm,MambaHead,measured,23963148,23963148,NaN,NaN,NaN,NaN,768.0,64.0,6.0,NaN,NaN
3,xlstm,xlstm,XLSTMHead,measured,24259180,24259180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,704.0,8.0
4,mlp_mixer,timm,LogitNormalizationWrapper,measured,24144108,24144108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.sort_values("model").to_csv("../outputs/model_parameter_counts.csv", index=False)
print("Saved ../outputs/model_parameter_counts.csv")

Saved ../outputs/model_parameter_counts.csv
